# id maps and merge (refer to src/diffusion_kernel_later_merge.ipynb)

In [1]:
import networkx as nx
import numpy as np
from scipy.linalg import expm
import pickle
import os
import mygene
import pandas as pd

## id maps and merge (from entrz to uniport)

In [2]:
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019.txt'

G = nx.read_edgelist(file_path, nodetype=str, create_using=nx.Graph())
ensps = list(G.nodes())
del G

ppi_ids_map = get_map_df(ensps,'entrezgene')
ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

more2one_df['string_id'] = more2one_df['query']
merge_dict = more2one_df.groupby('uniprot_ids')['string_id'].apply(list).to_dict()
map_dict = dict()
merge_groups = []
for key in merge_dict:
    merge_groups.append(merge_dict[key])
    new_key = '_'.join(sorted(merge_dict[key]))
    map_dict[new_key] = key
flat_set = {item for sublist in merge_groups for item in sublist}
unique_ids = [ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]
delete_list = list(set(ensps) - flat_set - set(unique_ids))
sample_names = ensps

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
54 input query terms found no hit:	['9220', '26148', '11217', '147791', '79686', '284014', '23285', '6025', '440414', '117153', '373073


In [8]:
len(merge_groups),len(flat_set),len(unique_ids)

(46, 119, 16150)

In [9]:
unique_df = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)][['query','uniprot_ids']]
unique_df['string_ids'] = unique_df['query']
unique_mapping_dict = unique_df.set_index('string_ids')['uniprot_ids'].to_dict()
map_dict.update(unique_mapping_dict)
len(map_dict)

16196

In [10]:
diffusion_uniports = [value for key, value in map_dict.items()]
len(diffusion_uniports)

16196

## merged df across all features

In [12]:
from features_reindex import get_feature
from sklearn.preprocessing import MinMaxScaler
merged_df = None

root = '/itf-fi-ml/shared/users/ziyuzh/svm'
time_feature_list = ['uniport_bio','uniport_seq','uniport_esm','biograd_2019_n2v']
feature_list = time_feature_list
for feature in time_feature_list:
    feature_df = get_feature(root, feature)

    feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
    if feature_cols:
        scaler = MinMaxScaler()
        feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)

    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory
name_list = feature_list + ['string_id']

merged_df = merged_df[[col for col in merged_df.columns if any(item in col for item in name_list)]]
len(merged_df)

14173

In [13]:
len(set(diffusion_uniports)-set(merged_df['string_id'].tolist())), len(set(merged_df['string_id'].tolist())-set(diffusion_uniports))

(2023, 0)

In [14]:
delete_uniports = set(diffusion_uniports)-set(merged_df['string_id'].tolist())

In [15]:
map_dict_aligned = dict()
mapped_ensp = []
for key, value in map_dict.items():
    if value in delete_uniports:
        pass
    else:
        map_dict_aligned[key] = value
        mapped_ensp.extend(key.split('_'))
len(map_dict_aligned),len(mapped_ensp)

(14173, 14241)

In [16]:
delete_ensp = list(set(ensps) - set(mapped_ensp))
len(delete_ensp),len(map_dict_aligned)


(3168, 14173)

In [17]:
recalculate = []
for key, value in map_dict_aligned.items():
    if '_' in key:
        recalculate.append(key.split('_'))
len(recalculate),recalculate[0]

(41, ['353513', '9084'])

In [21]:

save_path_1 = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019_map.pkl'
os.makedirs(os.path.dirname(save_path_1), exist_ok=True)
with open(save_path_1, 'wb') as f:
    pickle.dump([recalculate,delete_ensp,map_dict_aligned], f)

# run scr/diffusion.py

# merge df kernels to others (which is already calculated using model_biogrid.py)

In [1]:
import pickle
kernel_dict_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/path_save.pkl'
with open(kernel_dict_path, 'rb') as f:
    kernels_all_dict = pickle.load(f)

In [3]:
kernels_all_dict

{'uniport_bio': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_2_0.5485944881803674.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_2_0.5485944881803674.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_4_0.2742972440901837.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_4_0.2742972440901837.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_8_0.13714862204509184.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_8_0.13714862204509184.pkl']},
 'uniport_esm': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_K_2_0.11059481853189618.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_logK_2_0.11059481853189618.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid

In [11]:

# Set the directory where your files are located
directory = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019'

# Loop through all files in the directory
for filename in os.listdir(directory):
    if "difussion" in filename:
        new_filename = filename.replace("difussion", "diffusion")
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")

Renamed: uniport_ids_difussion_K_0.1.pkl -> uniport_ids_diffusion_K_0.1.pkl
Renamed: uniport_difussion_logK_2.pkl -> uniport_diffusion_logK_2.pkl
Renamed: uniport_difussion_logK_1.pkl -> uniport_diffusion_logK_1.pkl
Renamed: uniport_difussion_logK_0.5.pkl -> uniport_diffusion_logK_0.5.pkl
Renamed: uniport_ids_difussion_K_2.pkl -> uniport_ids_diffusion_K_2.pkl
Renamed: uniport_ids_difussion_K_0.8.pkl -> uniport_ids_diffusion_K_0.8.pkl
Renamed: uniport_ids_difussion_K_0.5.pkl -> uniport_ids_diffusion_K_0.5.pkl
Renamed: uniport_difussion_K_0.1.pkl -> uniport_diffusion_K_0.1.pkl
Renamed: uniport_difussion_logK_0.8.pkl -> uniport_diffusion_logK_0.8.pkl
Renamed: uniport_difussion_K_1.pkl -> uniport_diffusion_K_1.pkl
Renamed: uniport_difussion_K_0.5.pkl -> uniport_diffusion_K_0.5.pkl
Renamed: uniport_ids_difussion_K_0.2.pkl -> uniport_ids_diffusion_K_0.2.pkl
Renamed: uniport_difussion_logK_0.1.pkl -> uniport_diffusion_logK_0.1.pkl
Renamed: uniport_difussion_K_0.2.pkl -> uniport_diffusion_K_0.

In [3]:
import os
df_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019'
df_dict = dict()
for file in os.listdir(df_path):
    if 'uniport_diffusion_K' in file:
        temp_list = []
        key = file.split('_')[-1][:-4]
        temp_list.append(os.path.join(df_path,file))
        temp_list.append(os.path.join(df_path,'uniport_diffusion_logK_'+file.split('_')[-1]))
        df_dict[key] = temp_list

In [4]:
df_dict

{'0.2': ['/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_K_0.2.pkl',
  '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_logK_0.2.pkl'],
 '1': ['/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_K_1.pkl',
  '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_logK_1.pkl'],
 '0.5': ['/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_K_0.5.pkl',
  '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_logK_0.5.pkl'],
 '2': ['/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_K_2.pkl',
  '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_logK_2.pkl'],
 '0.1': ['/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_K_0.1.pkl',
  '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019/uniport_diffusion_logK_0.1.pkl'],
 '0.8': ['/itf-fi-ml/shared/users/ziy

In [5]:
kernels_all_dict = {'uniport_bio': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_2_0.5485944881803674.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_2_0.5485944881803674.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_4_0.2742972440901837.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_4_0.2742972440901837.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_K_8_0.13714862204509184.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_bio_logK_8_0.13714862204509184.pkl']},
 'uniport_esm': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_K_2_0.11059481853189618.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_logK_2_0.11059481853189618.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_K_4_0.05529740926594809.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_logK_4_0.05529740926594809.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_K_8_0.027648704632974044.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_esm_logK_8_0.027648704632974044.pkl']},
 'biograd_2019_n2v': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_K_2_0.4847661695471371.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_logK_2_0.4847661695471371.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_K_4_0.24238308477356854.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_logK_4_0.24238308477356854.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_K_8_0.12119154238678427.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_n2v_logK_8_0.12119154238678427.pkl']},
 'uniport_seq': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_K_2_0.22042045731460816.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_logK_2_0.22042045731460816.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_K_4_0.11021022865730408.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_logK_4_0.11021022865730408.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_K_8_0.05510511432865204.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/uniport_seq_logK_8_0.05510511432865204.pkl']}}

kernels_all_dict['biogrid_diffusion_2019'] = df_dict

In [2]:
kernels_all_dict.keys()

dict_keys(['uniport_bio', 'uniport_esm', 'biograd_2019_n2v', 'uniport_seq', 'biogrid_diffusion_2019'])

In [6]:
import os
for file in os.listdir('/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019'):
    if 'dw' in file:
        print(file)

biograd_2019_dw_40_K_8_0.10132842530684026.pkl
biograd_2019_dw_40_logK_4_0.20265685061368052.pkl
biograd_2019_dw_40_logK_8_0.10132842530684026.pkl
biograd_2019_dw_40_K_2_0.40531370122736105.pkl
biograd_2019_dw_40_logK_2_0.40531370122736105.pkl
biograd_2019_dw_40_K_4_0.20265685061368052.pkl


In [5]:
kernels_all_dict['biograd_2019_dw_40']={2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_K_2_0.40531370122736105.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_logK_2_0.40531370122736105.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_K_4_0.20265685061368052.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_logK_4_0.20265685061368052.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_K_8_0.10132842530684026.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/biograd_2019_dw_40_logK_8_0.10132842530684026.pkl']}

In [6]:
kernels_all_dict.keys()

dict_keys(['uniport_bio', 'uniport_esm', 'biograd_2019_n2v', 'uniport_seq', 'biogrid_diffusion_2019', 'biograd_2019_dw_40'])

In [7]:
kernel_dict_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/path_save.pkl'

with open(kernel_dict_path, 'wb') as f:
    pickle.dump(kernels_all_dict, f)

## create diffusion feature file (for main_biogrid.py)

In [9]:
import os
import pickle
kernel_pkl_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/biogrid_2019_all/2019/path_save.pkl'
df_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df_biogrid/2019'

if os.path.isfile(kernel_pkl_path):
    print('kernels existing')
    with open(kernel_pkl_path, 'rb') as f:
        kernels_all_dict = pickle.load(f)
else:
    kernels_all_dict = dict()
feature_list = ['diffusion_2019']
add_feature_list = set(feature_list) - set(kernels_all_dict.keys())

kernels existing


In [10]:
set(kernels_all_dict.keys())

{'biograd_2019_n2v',
 'biogrid_diffusion_2019',
 'uniport_bio',
 'uniport_esm',
 'uniport_seq'}

In [5]:
id_lists = []
for file in os.listdir(df_path):
    if 'uniport_ids' in file:
        id_path = os.path.join(df_path,file)
        with open(id_path, 'rb') as f:
            id_list = pickle.load(f)
        id_lists.append(id_list)
if id_lists[0] == id_lists[1]:
    print(1)

1


In [6]:
# check if the ids are same (including order), if they are same:
# Convert to DataFrame
import pandas as pd
df = pd.DataFrame({
    'string_id': id_lists[0],
    'feature_0': range(len(id_lists[0]))
})
df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/biogrid_diffusion_2019.csv',index=False)